# Feature Engineering - ISPU Forecasting (2021+)

**Objective:** Process merged data from 2021 onwards and create comprehensive features for ISPU forecasting.

**Steps:**
1. Load data from `dataset_processing/dataset/merged_data_v4_complete.csv`
2. Filter data from 2021-01-01 onwards
3. Create time-based features (cyclical encoding, seasons, holidays)
4. Create lag and rolling window features
5. Create pollutant ratio and interaction features
6. Save final feature set for modeling

In [3]:
# 1. Imports
import pandas as pd
import numpy as np
import warnings
import os

warnings.filterwarnings('ignore')
print("Setup complete.")

Setup complete.


In [4]:
# 2. Load Raw Data
DATA_PATH = '../dataset_processing/dataset/merged_data_v4_complete.csv'
# DATA_PATH = '../dataset/final_dataset_ready_for_modeling.csv'

df = pd.read_csv(DATA_PATH)

# Standardize column names
rename_map = {'pm_sepuluh': 'pm10', 'pm_duakomalima': 'pm25'}
df = df.rename(columns=rename_map)

df['tanggal'] = pd.to_datetime(df['tanggal'])
print(f"Loaded: {df.shape}")
df.head()

Loaded: (15412, 45)


,tanggal,stasiun,pm10,pm25,so2,co,o3,no2,max,critical_parameter,...,ventilation_proxy,rainy_day,is_holiday_nasional,nama_libur,is_weekend,day_name,is_working_day,tahun,Total_Penduduk,ndvi
0,2010-01-01,DKI1,60.0,NaN,4.0,73.0,27.0,14.0,73.0,CO,...,0.010422,1.0,1.0,New Year's Day,0.0,Friday,0,2010.0,902973.0,0.202300
1,2010-01-02,DKI1,32.0,NaN,2.0,16.0,33.0,9.0,33.0,O3,...,0.007638,1.0,0.0,NaN,1.0,Saturday,0,2010.0,902973.0,0.203619
2,2010-01-03,DKI1,27.0,NaN,2.0,19.0,20.0,9.0,27.0,PM10,...,0.009321,1.0,0.0,NaN,1.0,Sunday,0,2010.0,902973.0,0.204937
3,2010-01-04,DKI1,22.0,NaN,2.0,16.0,15.0,6.0,22.0,PM10,...,0.013400,0.0,0.0,NaN,0.0,Monday,1,2010.0,902973.0,0.206256
4,2010-01-05,DKI1,25.0,NaN,2.0,17.0,15.0,8.0,25.0,PM10,...,0.011015,1.0,0.0,NaN,0.0,Tuesday,1,2010.0,902973.0,0.207575


In [5]:
# 3. Preprocessing
df = df.sort_values(['stasiun', 'tanggal'])
df = df.set_index('tanggal')
print(f"Data shape: {df.shape}")

Data shape: (15412, 44)


In [6]:
# 4. Feature Engineering
TARGET = 'pm10'

def engineer_features(df_input):
    df_feat = df_input.copy()
    df_feat['dayofweek'] = df_feat.index.dayofweek
    df_feat['quarter'] = df_feat.index.quarter
    df_feat['month_feat'] = df_feat.index.month
    df_feat['year_feat'] = df_feat.index.year
    df_feat['dayofyear'] = df_feat.index.dayofyear
    
    df_feat['lag_1'] = df_feat.groupby('stasiun')[TARGET].shift(1)
    df_feat['lag_7'] = df_feat.groupby('stasiun')[TARGET].shift(7)
    df_feat['lag_30'] = df_feat.groupby('stasiun')[TARGET].shift(30)
    
    df_feat['rolling_mean_7'] = df_feat.groupby('stasiun')[TARGET].transform(
        lambda x: x.rolling(window=7, min_periods=1).mean())
    df_feat['rolling_std_7'] = df_feat.groupby('stasiun')[TARGET].transform(
        lambda x: x.rolling(window=7, min_periods=1).std())
    df_feat['rolling_mean_30'] = df_feat.groupby('stasiun')[TARGET].transform(
        lambda x: x.rolling(window=30, min_periods=1).mean())
    
    return df_feat

df_featured = engineer_features(df)
print(f"Data shape after feature engineering: {df_featured.shape}")

Data shape after feature engineering: (15412, 55)


In [7]:
# 5. Save to final_feature.csv
OUTPUT_PATH = 'final_feature_2021.csv'

# Reset index so 'tanggal' is a column again
df_featured = df_featured.reset_index()

df_featured.to_csv(OUTPUT_PATH, index=False)
print(f"Saved to {OUTPUT_PATH}")
print(f"Shape: {df_featured.shape}")
print(f"Columns: {list(df_featured.columns)}")

Saved to final_feature_2021.csv
Shape: (15412, 56)
Columns: ['tanggal', 'stasiun', 'pm10', 'pm25', 'so2', 'co', 'o3', 'no2', 'max', 'critical_parameter', 'target_kategori', 'temperature_2m_max', 'temperature_2m_min', 'precipitation_sum', 'precipitation_hours', 'wind_speed_10m_max', 'wind_direction_10m_dominant', 'shortwave_radiation_sum', 'temperature_2m_mean', 'relative_humidity_2m_mean', 'cloud_cover_mean', 'surface_pressure_mean', 'wind_gusts_10m_max', 'winddirection_10m_dominant', 'relative_humidity_2m_max', 'relative_humidity_2m_min', 'cloud_cover_max', 'cloud_cover_min', 'wind_gusts_10m_mean', 'wind_speed_10m_mean', 'wind_gusts_10m_min', 'wind_speed_10m_min', 'surface_pressure_max', 'surface_pressure_min', 'washout_effect', 'ventilation_proxy', 'rainy_day', 'is_holiday_nasional', 'nama_libur', 'is_weekend', 'day_name', 'is_working_day', 'tahun', 'Total_Penduduk', 'ndvi', 'dayofweek', 'quarter', 'month_feat', 'year_feat', 'dayofyear', 'lag_1', 'lag_7', 'lag_30', 'rolling_mean_7', 